<a href="https://colab.research.google.com/github/mohamedelguendy/Data-Integrity-and-Authentication/blob/main/Data_Integrity_and_Authentication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Sender Code (Student 1)

In [ ]:
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import serialization
import base64

# Generate keys
private_key = rsa.generate_private_key(
    public_exponent=65537,
    key_size=2048
)

public_key = private_key.public_key()

# User input
message = input("Enter your message: ").encode()

# Hash
digest = hashes.Hash(hashes.SHA256())
digest.update(message)
message_hash = digest.finalize()

# Sign
signature = private_key.sign(
    message_hash,
    padding.PSS(
        mgf=padding.MGF1(hashes.SHA256()),
        salt_length=padding.PSS.MAX_LENGTH
    ),
    hashes.SHA256()
)

# Convert to printable format
signature_b64 = base64.b64encode(signature).decode()
public_key_pem = public_key.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo
).decode()

# OUTPUT (to send manually)
print("\n=== SEND THIS TO RECEIVER ===")
print("Message:", message.decode())
print("Signature:", signature_b64)
print("Public Key:\n", public_key_pem)

Enter your message: Hello

=== SEND THIS TO RECEIVER ===
Message: Hello
Signature: Tm3P/VaCIwkmTRPtOvq+uqzAd4UMIUTXJrmAHgZY4fknRc7qtcI3h0Z6moeTKt1HqepnwCOm8/UAnRKxeB8xjV7dot0HZul9F08kTVU2Hov2ElGRcOcP7u6pRLeDbw0FGpUeuY4aW7xsdmlso7uTkm8SfGnbGxtMXF3IkMkQBeoVs37zsY+Xuub6Gfuw75tfoVdeBvD5ioZxE1QDnws1Nbg5uVnfnw96mpOgUm79cjZmJEEX8aA/Rz8jRQTCUjUpvM0EgRXFcWomYXVlhNABgzt27A46F1TAJBEN53uRsqdBqrFyfHSJkISfZrkDqs35VHj754NppZ8qaDob5h/zPw==
Public Key:
 -----BEGIN PUBLIC KEY-----
MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEApgCPw6+FLGxF7jgPNgYE
Od4K7LEr2g0A5TRt8EheqSjVGcfdbPNSyFWfE4DWNpQnm/4pWXYhjh4sS8o4xTBR
G41XnVz1B1eTPl5tqJLxIx6YPpG7aTwqz+V8ysDANKXHZeRU9nol/bbyVMLSth6e
QFOSPvjmP/mCuTIyuZoiLWCRUWBVn1/1XFpvq3RWQlqO1cjm3yBnRv1/rLbN1eB2
Q7Jr64tDFTQucZNOaU4z7ZJF+cuc7pG3nRwDaLwEh8sKSuyS540DN2kKQJWwsKel
zLIy/GCHK+eUwwI3mK4wFFxraXx+g5Vu0ni2kJg6UR9EI5PPlKkZ7r3EUL/Havlq
7QIDAQAB
-----END PUBLIC KEY-----



### Receiver Code (Student 2)

In [ ]:
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import padding
import base64

# ===== INPUT from Sender =====
message_input = input("Enter received message: ")
signature_input = input("Enter received signature: ")

print("Paste public key (end with empty line):")
lines = []
while True:
    line = input()
    if line == "":
        break
    lines.append(line)
public_key_input = "\n".join(lines)

# Convert inputs
message = message_input.encode()
signature = base64.b64decode(signature_input)
public_key = serialization.load_pem_public_key(public_key_input.encode())

# Hash again
digest = hashes.Hash(hashes.SHA256())
digest.update(message)
receiver_hash = digest.finalize()

# Verify
try:
    public_key.verify(
        signature,
        receiver_hash,
        padding.PSS(
            mgf=padding.MGF1(hashes.SHA256()),
            salt_length=padding.PSS.MAX_LENGTH
        ),
        hashes.SHA256()
    )

    print("\n VALID: Message is authentic and not modified")

except Exception:
    print("\n INVALID: Message was modified or fake")

Enter received message: hellooooo
Enter received signature: Tm3P/VaCIwkmTRPtOvq+uqzAd4UMIUTXJrmAHgZY4fknRc7qtcI3h0Z6moeTKt1HqepnwCOm8/UAnRKxeB8xjV7dot0HZul9F08kTVU2Hov2ElGRcOcP7u6pRLeDbw0FGpUeuY4aW7xsdmlso7uTkm8SfGnbGxtMXF3IkMkQBeoVs37zsY+Xuub6Gfuw75tfoVdeBvD5ioZxE1QDnws1Nbg5uVnfnw96mpOgUm79cjZmJEEX8aA/Rz8jRQTCUjUpvM0EgRXFcWomYXVlhNABgzt27A46F1TAJBEN53uRsqdBqrFyfHSJkISfZrkDqs35VHj754NppZ8qaDob5h/zPw==
Paste public key (end with empty line):
-----BEGIN PUBLIC KEY----- MIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEApgCPw6+FLGxF7jgPNgYE Od4K7LEr2g0A5TRt8EheqSjVGcfdbPNSyFWfE4DWNpQnm/4pWXYhjh4sS8o4xTBR G41XnVz1B1eTPl5tqJLxIx6YPpG7aTwqz+V8ysDANKXHZeRU9nol/bbyVMLSth6e QFOSPvjmP/mCuTIyuZoiLWCRUWBVn1/1XFpvq3RWQlqO1cjm3yBnRv1/rLbN1eB2 Q7Jr64tDFTQucZNOaU4z7ZJF+cuc7pG3nRwDaLwEh8sKSuyS540DN2kKQJWwsKel zLIy/GCHK+eUwwI3mK4wFFxraXx+g5Vu0ni2kJg6UR9EI5PPlKkZ7r3EUL/Havlq 7QIDAQAB -----END PUBLIC KEY-----


❌ INVALID: Message was modified or fake
